# Prepare Beauty Atomic Files


In [21]:
from pathlib import Path
import pandas as pd
import glob
import numpy as np
import torch

In [22]:
# --- Config ---
BEAUTY_CATEGORY = "Beauty_and_Personal_Care"
DATA_DIR: str = "../data"

# Data split cutoff dates
TRAIN_END_CUTOFF_DATE: str = "2022-08-01"
VALID_END_CUTOFF_DATE: str = "2022-10-01"

# User and item thresholds
USER_MIN_REVIEWS: int = 5
WARM_USER_MIN_REVIEWS: int = 10
WARM_ITEM_MIN_REVIEWS: int = 5

# Downsampling for faster iteration
RANDOM_SEED: int = 42
MAX_TRAIN_SIZE: int | None = None
MAX_VALID_SIZE: int | None = None
MAX_TEST_SIZE: int | None = None

# Image embeddings
EMBEDDINGS_DIR = "../data/embeddings"
CLIP_DIM = 512


## Load reviews


In [3]:
df_reviews = pd.read_csv(f"{DATA_DIR}/reviews.csv")
display(df_reviews.sample(10))
df_reviews.info()


,user_id,parent_asin,rating,timestamp,category
8649655,AFVZJZMJEJKDVFP4HZ5EWYMLZWBA,B07DRBHRLW,5.0,1627047955216,Clothing_Shoes_and_Jewelry
25172077,AHLYHSYKZPT2Z2DFWY6E3NHXBBDA,B0BB5YD2WC,5.0,1668629109707,Beauty_and_Personal_Care
20898864,AETVUZEQDRLX45WJ7ELIK6MN7FHA,B09TPQ1YJG,5.0,1658857966758,Clothing_Shoes_and_Jewelry
23882129,AHYSF3GJCISOLAYTRWP6WXE3KX3A,B07BKNDST9,4.0,1665775746542,Clothing_Shoes_and_Jewelry
23805916,AF24NMN257GOUQNNEZWEFNX5CO6Q,B0BH9FD983,5.0,1665614849387,Beauty_and_Personal_Care
14363294,AEGDWFP7OFRSNPRYLS7PSDV52Q4A,B0B218ZGTP,5.0,1642002427220,Clothing_Shoes_and_Jewelry
13384324,AGUTEVXC7IJ6PHCC3ZBFK7TESHZQ,B086CVRMQG,5.0,1639754688974,Clothing_Shoes_and_Jewelry
25625087,AFIH6ASJPYWLQ644PLBKYHMH2VXQ,B07DXPZN3H,5.0,1669792885631,Clothing_Shoes_and_Jewelry
11353872,AHCKRJUYHU4T7AOJB6J6FGCDOGCQ,B09PG9BN5G,4.0,1634002228304,Clothing_Shoes_and_Jewelry
23967911,AEX5NRWBBCOZBOJD6OMQQXCPZCTA,B084Q6FNNC,5.0,1665975878755,Clothing_Shoes_and_Jewelry


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26876611 entries, 0 to 26876610
Data columns (total 5 columns):
 #   Column       Dtype  
---  ------       -----  
 0   user_id      object 
 1   parent_asin  object 
 2   rating       float64
 3   timestamp    int64  
 4   category     object 
dtypes: float64(1), int64(1), object(3)
memory usage: 1.0+ GB


## Load items


In [4]:
df_items = pd.read_csv(f"{DATA_DIR}/items.csv")
display(df_items.sample(10))
df_items.info()


,parent_asin,title,price,store,category
1577675,B0B77ZGSKG,"Upgraded Watch Link Removal Tool Kit, Watch Ba...",NaN,VCCGY,Clothing_Shoes_and_Jewelry
2169137,B08N12LPPZ,WDIRARA Women's Leopard Heart Print Round Neck...,NaN,WDIRARA,Clothing_Shoes_and_Jewelry
919029,B0826VQHNG,BRIEF INSANITY We The People Boxer Briefs for ...,20.00,BRIEF INSANITY,Clothing_Shoes_and_Jewelry
683592,B08FHJGGWT,Women Fashion Snakeskin Leopard Zebra Print Ho...,11.54,HuiApparel,Clothing_Shoes_and_Jewelry
1283618,B0923W4DQF,DAZCOS Venti Barbatos Cosplay Hair Clip The Ce...,8.99,DAZCOS,Clothing_Shoes_and_Jewelry
459030,B078PY1D34,Basil Shower Gel/8.5 oz.,29.99,Le Labo,Beauty_and_Personal_Care
2198151,B09VJFS94F,Plaid Hooded Plush Robes for Women and Men - P...,NaN,Personalized Passion,Clothing_Shoes_and_Jewelry
1924098,B01IAS5LVG,Sanuk Men's Bandito Flip Flop,NaN,Sanuk,Clothing_Shoes_and_Jewelry
395330,B09WR1J48C,"AUBSS 15ml Gel Base and Top Coat Set, Gel Top ...",NaN,AUBSS,Beauty_and_Personal_Care
1898585,B08KFKXRJ9,Fly Flot Women's Flat Slipper,NaN,Fly Flot,Clothing_Shoes_and_Jewelry


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2694121 entries, 0 to 2694120
Data columns (total 5 columns):
 #   Column       Dtype  
---  ------       -----  
 0   parent_asin  object 
 1   title        object 
 2   price        float64
 3   store        object 
 4   category     object 
dtypes: float64(1), object(4)
memory usage: 102.8+ MB


## Map user/item IDs to integers


In [5]:
# As RecBole expects integer IDs, we need to create mappings from the original string IDs to integers.
user_ids: set[str] = set(df_reviews["user_id"])
item_ids: set[str] = set(df_reviews["parent_asin"])

user_map: dict[str, int] = {uid: i+1 for i, uid in enumerate(sorted(user_ids))}
item_map: dict[str, int] = {pid: i+1 for i, pid in enumerate(sorted(item_ids))}

In [6]:
df_reviews['uid'] = df_reviews['user_id'].map(user_map)
df_reviews['iid'] = df_reviews['parent_asin'].map(item_map)
df_items['iid'] = df_items['parent_asin'].map(item_map)


## Filter to beauty category


In [7]:
df_reviews = df_reviews[df_reviews["category"] == BEAUTY_CATEGORY]
print(f"Beauty reviews: {len(df_reviews):,}")

Beauty reviews: 7,300,381


## Filter min user reviews

In [8]:
before_users = df_reviews["uid"].nunique()
user_counts = df_reviews.groupby("uid").size()
valid_users = user_counts[user_counts >= USER_MIN_REVIEWS].index
df_reviews = df_reviews[df_reviews["uid"].isin(valid_users)]
after_users = df_reviews["uid"].nunique()
print(f"Removed {before_users - after_users:,} users with < {USER_MIN_REVIEWS} reviews "
      f"({after_users:,} users remain, {len(df_reviews):,} reviews)")

Removed 4,221,111 users with < 5 reviews (168,377 users remain, 1,585,905 reviews)


## Split and filter train/valid/test 


In [9]:
def _date_to_ms(date_str: str) -> int:
    return int(pd.Timestamp(date_str, tz="UTC").timestamp() * 1000)

timestamps = sorted(df_reviews["timestamp"].values)
train_start_ts = timestamps[0]
train_end_ts = _date_to_ms(TRAIN_END_CUTOFF_DATE)
valid_end_ts = _date_to_ms(VALID_END_CUTOFF_DATE)
print(f"Train start timestamp: {train_start_ts} ({pd.Timestamp(train_start_ts, unit='ms', tz='UTC')})")
print(f"Train end timestamp: {train_end_ts} ({pd.Timestamp(train_end_ts, unit='ms', tz='UTC')})")
print(f"Valid end timestamp: {valid_end_ts} ({pd.Timestamp(valid_end_ts, unit='ms', tz='UTC')})")


Train start timestamp: 1609459264769 (2021-01-01 00:01:04.769000+00:00)
Train end timestamp: 1659312000000 (2022-08-01 00:00:00+00:00)
Valid end timestamp: 1664582400000 (2022-10-01 00:00:00+00:00)


In [10]:
df_train = df_reviews[df_reviews["timestamp"] <= train_end_ts]
df_valid = df_reviews[(df_reviews["timestamp"] > train_end_ts) & (df_reviews["timestamp"] <= valid_end_ts)]
df_test = df_reviews[df_reviews["timestamp"] > valid_end_ts]

print(f"Train reviews: {len(df_train):,}")
print('Train interactions per user', len(df_train) / len(df_train['uid'].unique()))

if MAX_TRAIN_SIZE is not None and len(df_train) > MAX_TRAIN_SIZE:
    df_train = df_train.sample(n=MAX_TRAIN_SIZE, random_state=RANDOM_SEED)
    print(f"Downsampled train reviews to {len(df_train):,}")
if MAX_VALID_SIZE is not None and len(df_valid) > MAX_VALID_SIZE:
    df_valid = df_valid.sample(n=MAX_VALID_SIZE, random_state=RANDOM_SEED)
    print(f"Downsampled valid reviews to {len(df_valid):,}")
if MAX_TEST_SIZE is not None and len(df_test) > MAX_TEST_SIZE:
    df_test = df_test.sample(n=MAX_TEST_SIZE, random_state=RANDOM_SEED)
    print(f"Downsampled test reviews to {len(df_test):,}")

train_users = set(df_train["uid"].unique())
df_valid = df_valid[df_valid["uid"].isin(train_users)]
df_test = df_test[df_test["uid"].isin(train_users)]
df_all = pd.concat([df_train, df_valid, df_test])

print(f"Valid reviews: {len(df_valid):,}")
print(f"Test reviews: {len(df_test):,}")

display(pd.DataFrame({
    "split": ["train", "valid", "test"],
    "reviews": [len(df_train), len(df_valid), len(df_test)],
    "users": [df_train["uid"].nunique(), df_valid["uid"].nunique(), df_test["uid"].nunique()],
    "items": [df_train["iid"].nunique(), df_valid["iid"].nunique(), df_test["iid"].nunique()],
}))

Train reviews: 1,125,023
Train interactions per user 7.18924255688971
Valid reviews: 139,338
Test reviews: 197,706


,split,reviews,users,items
0,train,1125023,156487,215358
1,valid,139338,47209,52826
2,test,197706,55594,66743


## Define cold vs. warm users/items with train data


In [11]:
df_train_users = df_train.groupby("uid").size().reset_index(name="num_train")
cold_user_ids: set[int] = set(df_train_users[df_train_users["num_train"] < WARM_USER_MIN_REVIEWS]["uid"])
warm_user_ids: set[int] = set(df_train_users[df_train_users["num_train"] >= WARM_USER_MIN_REVIEWS]["uid"])
print(f"Warm users (>= {WARM_USER_MIN_REVIEWS} reviews): {len(warm_user_ids):,}")
print(f"Cold users (< {WARM_USER_MIN_REVIEWS} reviews): {len(cold_user_ids):,}")


Warm users (>= 10 reviews): 20,100
Cold users (< 10 reviews): 136,387


In [12]:
## Define cold vs. warm items with train data
df_train_items = df_train.groupby("iid").size().reset_index(name="num_train")
cold_item_ids: set[int] = set(df_train_items[df_train_items["num_train"] < WARM_ITEM_MIN_REVIEWS]["iid"])
warm_item_ids: set[int] = set(df_train_items[df_train_items["num_train"] >= WARM_ITEM_MIN_REVIEWS]["iid"])
print(f"Warm items (>= {WARM_ITEM_MIN_REVIEWS} train reviews): {len(warm_item_ids):,}")
print(f"Cold items (< {WARM_ITEM_MIN_REVIEWS} train reviews): {len(cold_item_ids):,}")


Warm items (>= 5 train reviews): 50,483
Cold items (< 5 train reviews): 164,875


In [17]:
all_user_ids = set(df_all['uid'].unique())
all_item_ids = df_items['iid'].isin(df_all['iid'].unique())
print(f"Unique users in all splits: {len(all_user_ids):,}")
print(f"Unique items in all splits: {all_item_ids.sum():,}")

Unique users in all splits: 156,487
Unique items in all splits: 250,852


## Write atomic files


In [18]:
def get_user_category(uid: int) -> int:
    if uid in warm_user_ids:
        return 0  # Warm user
    return 1      # Cold user

def write_user_file(path: Path, uids: set[int]) -> None:
    rows = [(uid, get_user_category(uid)) for uid in uids]
    df_out = pd.DataFrame(rows, columns=["user_id:token", "cold:float"])
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")

def write_item_file(path: Path, df_items_subset: pd.DataFrame) -> None:
    df_out = df_items_subset[["iid", "title", "store", "price"]].copy()
    df_out["title"] = df_out["title"].fillna("").astype(str).str.replace('"', "", regex=False)
    df_out["store"] = df_out["store"].fillna("").astype(str).str.replace('"', "", regex=False)
    df_out["price"] = pd.to_numeric(df_out["price"], errors="coerce").fillna("")
    df_out["cold"] = df_out["iid"].apply(lambda iid: 1 if iid not in warm_item_ids else 0)
    df_out.columns = ["item_id:token", "title:token", "store:token", "price:float", "cold:float"]
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")

def write_inter_file(path: Path, df: pd.DataFrame) -> None:
    df_out = df[["uid", "iid", "rating", "timestamp"]].copy()
    df_out.columns = ["user_id:token", "item_id:token", "rating:float", "timestamp:float"]
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")


In [19]:
dataset_prefix = Path(DATA_DIR) / "beauty" / "beauty"
dataset_prefix.parent.mkdir(parents=True, exist_ok=True)
write_inter_file(dataset_prefix.with_suffix(".train.inter"), df_train)
write_inter_file(dataset_prefix.with_suffix(".valid.inter"), df_valid)
write_inter_file(dataset_prefix.with_suffix(".test.inter"), df_test)
write_user_file(dataset_prefix.with_suffix(".user"), all_user_ids)
write_item_file(dataset_prefix.with_suffix(".item"), df_items[all_item_ids])

Wrote ../data/beauty/beauty.train.inter (1,125,023 rows)
Wrote ../data/beauty/beauty.valid.inter (139,338 rows)
Wrote ../data/beauty/beauty.test.inter (197,706 rows)
Wrote ../data/beauty/beauty.user (156,487 rows)
Wrote ../data/beauty/beauty.item (250,852 rows)


## Save image embeddings

In [27]:
shard_paths = sorted(glob.glob(f"{EMBEDDINGS_DIR}/*.parquet"))
print(f"Found {len(shard_paths)} CLIP embedding shard(s)")

dfs = [pd.read_parquet(p) for p in shard_paths]
df_clip = pd.concat(dfs, ignore_index=True).drop_duplicates(subset="parent_asin", keep="last")
print(f"Loaded {len(df_clip):,} CLIP image embeddings (deduplicated by parent_asin)")

asin_to_clip: dict[str, np.ndarray] = dict(zip(
    df_clip["parent_asin"],
    df_clip["embedding"].apply(lambda x: np.asarray(x, dtype=np.float32)),
))


Found 914 CLIP embedding shard(s)
Loaded 4,380,169 CLIP image embeddings (deduplicated by parent_asin)


In [28]:
dataset_items = df_items[all_item_ids].copy()
n_items = len(dataset_items)
embs = np.zeros((n_items, CLIP_DIM), dtype=np.float32)
iid_to_idx: dict[int, int] = {}

n_found = 0
for idx, (_, row) in enumerate(dataset_items.iterrows()):
    iid = int(row["iid"])
    vec = asin_to_clip.get(row["parent_asin"])
    if vec is not None:
        embs[idx] = vec
        n_found += 1
    iid_to_idx[iid] = idx

torch.save(
    {"embeddings": torch.from_numpy(embs), "iid_to_idx": iid_to_idx},
    f"{DATA_DIR}/beauty/clip_image_embeddings.pt",
)
print(f"Saved CLIP image embeddings: {n_found:,} / {n_items:,} items have images")

Saved CLIP image embeddings: 250,822 / 250,852 items have images
